# Lab 05: Shuffle and joins (reference solution)

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>3.1 What is a shuffle?</b></span><br><span style="color:#d93025"><b>3.2 When is a broadcast join triggered?</b></span><br><span style="color:#d93025"><b>3.3 Cost of a shuffle</b></span><br><span style="color:#d93025"><b>3.4 Avoiding a shuffle in a join</b></span><br><span style="color:#d93025"><b>3.5 Bucketing</b></span><br><span style="color:#d93025"><b>3.6 Sort-merge join</b></span><br><span style="color:#d93025"><b>3.7 Hash join</b></span><br><span style="color:#d93025"><b>3.8 Cartesian product</b></span><br><span style="color:#d93025"><b>3.9 How Spark chooses a join strategy</b></span><br><span style="color:#d93025"><b>3.10 AQE for joins</b></span><br><span style="color:#d93025"><b>3.12 Broadcast hash vs. shuffle hash join</b></span>

**Goal:** measure what a shuffle costs, predict which join strategy Spark picks, and remove a shuffle on purpose. The last section is join practice.

## Data information
A small online shop. The setup cell below creates three tables in `data/` (seeded, so everyone gets the same rows):

**orders** (`data/orders.parquet`, 1,000,000 rows, January to March 2024)
* *order_id*, *customer_id* (empty for guest checkout), *product_id*, *quantity* (1 to 5)
* *amount*: quantity × product price, in dollars
* *status*: paid, shipped, cancelled or returned; *payment_method*: card, cash or wallet
* *order_ts*, *order_date*: when the order was placed
* *tracking_code*: a 32-character code

**customers** (`data/customers.csv`, 10,000 rows): *customer_id*, *name*, *country*, *signup_date*, *segment* (retail or business). Customer 1 is "Big Reseller Ltd", who places 30% of all orders.

**products** (`data/products.csv`, 200 rows): *product_id*, *name*, *category*, *price*

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [RDD guide: Shuffle operations](https://spark.apache.org/docs/latest/rdd-programming-guide.html#shuffle-operations)
* [Spark UI: Stages tab](https://spark.apache.org/docs/latest/web-ui.html#stages-tab)
* [Join strategy hints and the join strategy](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-the-join-strategy)
* [Join hints](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-hints.html)
* [JoinSelection (Japila)](https://books.japila.pl/spark-sql-internals/execution-planning-strategies/JoinSelection/)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [DataFrame.hint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html)
* [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution)
* [Bucketing, sorting and partitioning](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html#bucketing-sorting-and-partitioning)
* [DataFrameWriter.bucketBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.bucketBy.html)
* [SQL JOIN reference](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html)
* [functions.countDistinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.count_distinct.html)
* [functions.month](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.month.html)
* [Column.eqNullSafe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.eqNullSafe.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_customers(spark):
    c = spark.range(1, N_CUSTOMERS + 1).withColumnRenamed("id", "customer_id")
    return c.select(
        "customer_id",
        F.when(F.col("customer_id") == 1, "Big Reseller Ltd")
         .otherwise(F.concat(F.lit("Customer "), F.col("customer_id").cast("string"))).alias("name"),
        _pick(_u(F.col("customer_id"), 1), COUNTRIES).alias("country"),
        F.date_add(F.lit("2022-01-01").cast("date"), F.floor(_u(F.col("customer_id"), 2) * 730).cast("int")).alias("signup_date"),
        F.when((F.col("customer_id") == 1) | (_u(F.col("customer_id"), 3) < 0.2), "business").otherwise("retail").alias("segment"),
    )

def make_products(spark):
    p = spark.range(1, N_PRODUCTS + 1).withColumnRenamed("id", "product_id")
    return p.select(
        "product_id",
        F.concat(F.lit("Product "), F.col("product_id").cast("string")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.col("product_id") % 8 + 1).cast("int")).alias("category"),
        F.round((F.col("product_id") * 37) % 195 + 5.99, 2).alias("price"),
    )

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    customer = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                    # guest checkout: no customer
                 .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                       # the big reseller: 30% of orders
                 .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    quantity = F.when(customer == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), customer.alias("customer_id"), product.alias("product_id"), quantity.alias("quantity"),
        F.round(quantity * (((product * 37) % 195) + 5.99), 2).alias("amount"),
        status.alias("status"), payment.alias("payment_method"), ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "product_id", "quantity", "amount", "status",
                     "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_customers(spark).toPandas().to_csv("data/customers.csv", index=False)
    make_products(spark).toPandas().to_csv("data/products.csv", index=False)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day05_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
if not all(os.path.exists(f) for f in ("data/orders.parquet", "data/customers.csv", "data/products.csv", "data/monthly/orders_2024-03.parquet")):
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "5.2": "f820263018058ed9",
 "5.3": "901e4ffad976ca42",
 "5.4": "5ddd26300d32eb44",
 "5.5": "d05670f96e9058d8",
 "5.6": "3c4a3050de9c005b",
 "5.7": "3dd389b08a53d7e8",
 "5.8": "549efcbf1c83e669",
 "5.9": "2259aad091c166b0",
 "5.10": "8d3f7992197886d9"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the tables the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")
customers = (spark.read.option("header", True)
             .schema("customer_id long, name string, country string, signup_date date, segment string")
             .csv("data/customers.csv"))
products = (spark.read.option("header", True)
            .schema("product_id long, name string, category string, price double")
            .csv("data/products.csv"))
for name, df in [("orders", orders), ("customers", customers), ("products", products)]:
    df.createOrReplaceTempView(name)   # so SQL can use the tables too

## A. What a shuffle costs

**5.1 How much data does a shuffle move?** Measure the shuffle bytes written (`shuffle_bytes`) by `orders.groupBy("customer_id").count()` (*bytes_customer*) and by `orders.groupBy("status").count()` (*bytes_status*), both written with `noop`. Why is one so much smaller?

In [ ]:
bytes_customer = shuffle_bytes(lambda: noop(orders.groupBy("customer_id").count()))
bytes_status = shuffle_bytes(lambda: noop(orders.groupBy("status").count()))
bytes_customer, bytes_status

In [ ]:
check_true("5.1", bytes_customer > 10 * bytes_status, "grouping by a column with 10,000 values shuffles far more than grouping by 4 values (partial aggregation shrinks it first)")

## B. Join types and which side can be broadcast

**5.2 What does each join type return, and which side can Spark broadcast for it?** *left* has 4 rows and *right* has 5 (ids 1 to 4 and 3 to 7). Join them on *id* with the types `inner`, `left`, `right`, `full`, `left_semi` and `left_anti`.
* *row_counts* = dict `{type: number of rows}` (also `cross` for `crossJoin`)
* *honours* = dict `{type: (strategy with F.broadcast(right), strategy with F.broadcast(left))}`: the join strategy in the plan when you hint the right table, and when you hint the left table

Guess first. (Tables built from Python lists have no size estimate, so without a hint Spark never broadcasts them.)

In [ ]:
left = spark.createDataFrame([(1, "Ann"), (2, "Bob"), (3, "Cy"), (4, "Di")], "id int, name string")
right = spark.createDataFrame([(3, "x"), (4, "y"), (5, "z"), (6, "w"), (7, "v")], "id int, tag string")
def strategy_of(df):
    c = count_nodes(df, "BroadcastHashJoin", "ShuffledHashJoin", "SortMergeJoin", "BroadcastNestedLoopJoin", "CartesianProduct")
    return [s for s, n in c.items() if n][0]

row_counts, honours = {}, {}
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    row_counts[how] = left.join(right, "id", how).count()
    honours[how] = (strategy_of(left.join(F.broadcast(right), "id", how)), strategy_of(F.broadcast(left).join(right, "id", how)))
row_counts["cross"] = left.crossJoin(right).count()
row_counts, honours

In [ ]:
check("5.2", [row_counts, honours])

**5.3 Predict, then verify: which join strategy does Spark pick?** Put the strategy name of each case in a dict *strategies*:
* `a_orders_customers`: `orders.join(customers, "customer_id")`
* `b_two_big_tables`: `orders.select("order_id", "amount")` joined with `orders.select("order_id", "status")` on *order_id*
* `c_small_left_side`: `customers.join(orders, "customer_id", "left")`
* `d_full_outer`: orders and customers, `full` join
* `e_non_equi`: `orders.join(customers, orders.amount < customers.customer_id)`
* `f_non_equi_no_broadcast`: case e with `spark.sql.autoBroadcastJoinThreshold` set to -1
* `g_prefer_hash_join`: case a with the threshold set to `100KB` and `spark.sql.join.preferSortMergeJoin` set to false

Use `strategy_of` (given). Reset both settings afterwards (`spark.conf.unset`).

In [ ]:
def strategy_of(df):
    c = count_nodes(df, "BroadcastHashJoin", "ShuffledHashJoin", "SortMergeJoin", "BroadcastNestedLoopJoin", "CartesianProduct")
    return [s for s, n in c.items() if n][0]

two_big = orders.select("order_id", "amount").join(orders.select("order_id", "status"), "order_id")
strategies = {
    "a_orders_customers": strategy_of(orders.join(customers, "customer_id")),
    "b_two_big_tables": strategy_of(two_big),
    "c_small_left_side": strategy_of(customers.join(orders, "customer_id", "left")),
    "d_full_outer": strategy_of(orders.join(customers, "customer_id", "full")),
    "e_non_equi": strategy_of(orders.join(customers, orders.amount < customers.customer_id)),
}
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
strategies["f_non_equi_no_broadcast"] = strategy_of(orders.join(customers, orders.amount < customers.customer_id))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "100KB")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
strategies["g_prefer_hash_join"] = strategy_of(orders.join(customers, "customer_id"))
spark.conf.unset("spark.sql.join.preferSortMergeJoin")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
strategies

In [ ]:
check("5.3", strategies)

**5.4 Force each strategy with a hint and compare the cost.** Join *orders* with `customers.hint(h)` on *customer_id* for the hints `broadcast`, `shuffle_hash` and `merge`. Put the strategy of each in *hint_strategies* (`{hint: strategy}`) and the shuffle bytes written when running it with `noop` in *bytes_by* (`{hint: bytes}`).

In [ ]:
def with_hint(h):
    return orders.join(customers.hint(h), "customer_id")

hints = ["broadcast", "shuffle_hash", "merge"]
hint_strategies = {h: strategy_of(with_hint(h)) for h in hints}
bytes_by = {h: shuffle_bytes(lambda: noop(with_hint(h))) for h in hints}
hint_strategies, bytes_by

In [ ]:
check("5.4", hint_strategies)
check_true("5.4b", bytes_by["broadcast"] == 0 and bytes_by["shuffle_hash"] > 1e6 and bytes_by["merge"] > 1e6, "a broadcast join shuffles nothing; the other two shuffle both sides")

**5.5 AQE switches a join at runtime.** *recent* has the two columns of the orders of `2024-03-31` (about 11,000 rows). Build `q` = *orders* joined with *recent* on *order_id*, grouped by *status* with `count("recent_code")`.
* *static_strategy*: the strategy with AQE off (Spark thinks *recent* is big: filters do not shrink size estimates)
* then turn AQE on, build *q* again, run it with `collect()` and read the final plan (`plan(q, "simple")`, the part before `== Initial Plan ==`): *became_broadcast* (is there a `BroadcastHashJoin`?) and *local_reader* (does `AQEShuffleRead local` appear?)

Put `[static_strategy, became_broadcast, local_reader]` in *result*, then turn AQE off again.

In [ ]:
recent = (orders.filter(F.col("order_date") == "2024-03-31")
          .select("order_id", F.col("tracking_code").alias("recent_code")))

def build():
    return orders.join(recent, "order_id").groupBy("status").agg(F.count("recent_code").alias("n"))

static_strategy = strategy_of(build())
spark.conf.set("spark.sql.adaptive.enabled", "true")
q = build()
q.collect()
final = plan(q, "simple").split("== Initial Plan ==")[0]
result = [static_strategy, "BroadcastHashJoin" in final, "AQEShuffleRead local" in final]
spark.conf.set("spark.sql.adaptive.enabled", "false")
result

In [ ]:
check("5.5", result)

**5.6 A join with no shuffle: bucketing.** Turn broadcast joins off (`autoBroadcastJoinThreshold` = -1). Save `orders.select("order_id", "status", "amount")` as three bucketed tables on *order_id* (`bucketBy(n, "order_id").sortBy("order_id")`, `saveAsTable`): `o16`, `o16b` (16 buckets each) and `o8` (8 buckets). Count the shuffle `Exchange` nodes of these joins and put them in *exchanges* as a list:
* `o16` with `o16b` on *order_id*
* `o16` with `o8` on *order_id*
* the same, with `spark.sql.bucketing.coalesceBucketsInJoin.enabled` set to true
* `o16` with `o16b` on *amount* (not the bucket column)

Guess first, then reset the settings.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
for name, buckets in [("o16", 16), ("o16b", 16), ("o8", 8)]:
    (orders.select("order_id", "status", "amount").write.bucketBy(buckets, "order_id").sortBy("order_id")
           .mode("overwrite").saveAsTable(name))
ex = lambda df: count_nodes(df, "Exchange")["Exchange"]
t16, t16b, t8 = spark.table("o16"), spark.table("o16b"), spark.table("o8")
exchanges = [ex(t16.join(t16b, "order_id")), ex(t16.join(t8, "order_id"))]
spark.conf.set("spark.sql.bucketing.coalesceBucketsInJoin.enabled", "true")
exchanges.append(ex(t16.join(t8, "order_id")))
spark.conf.set("spark.sql.bucketing.coalesceBucketsInJoin.enabled", "false")
exchanges.append(ex(t16.join(t16b, t16.amount == t16b.amount)))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
exchanges

In [ ]:
check("5.6", exchanges)

## C. Coding practice: joins

Four problems: anti join, "in every month", range join and missing keys. Write them from scratch.

**5.7 Which customers never returned an order?** Use a `left_anti` join of *customers* with the customers that have a *returned* order. Put `[number of such customers, the 3 smallest customer_ids]` in *result*.

In [ ]:
returned_customers = orders.filter(F.col("status") == "returned").select("customer_id").distinct()
no_returns = customers.join(returned_customers, "customer_id", "left_anti")
result = [no_returns.count(), [r["customer_id"] for r in no_returns.orderBy("customer_id").limit(3).collect()]]
result

In [ ]:
check("5.7", result)

**5.8 Which customers returned orders in all three months (January, February and March)?** Ignore guests. Put `[number of such customers, the 3 smallest customer_ids]` in *result*.

In [ ]:
months = (orders.filter((F.col("status") == "returned") & F.col("customer_id").isNotNull())
          .groupBy("customer_id").agg(F.countDistinct(F.month("order_date")).alias("months")))
all_three = months.filter(F.col("months") == 3)
result = [all_three.count(), [r["customer_id"] for r in all_three.orderBy("customer_id").limit(3).collect()]]
result

In [ ]:
check("5.8", result)

**5.9 Put every order into a price band.** *bands* (given) has the columns *lo*, *hi* and *band*. Join *orders* with `F.broadcast(bands)` on the condition `lo <= amount < hi` (a range join) and count the orders per band into a dict *band_counts*. Which join strategy does Spark use (*join_name*)? Put `[band_counts, join_name]` in *result*.

In [ ]:
bands = spark.createDataFrame([(0.0, 50.0, "budget"), (50.0, 200.0, "standard"), (200.0, 500.0, "premium"), (500.0, 100000.0, "luxury")],
                              "lo double, hi double, band string")
def strategy_of(df):
    c = count_nodes(df, "BroadcastHashJoin", "ShuffledHashJoin", "SortMergeJoin", "BroadcastNestedLoopJoin", "CartesianProduct")
    return [s for s, n in c.items() if n][0]

banded = orders.join(F.broadcast(bands), (orders.amount >= bands.lo) & (orders.amount < bands.hi))
band_counts = {r["band"]: r["count"] for r in banded.groupBy("band").count().collect()}
join_name = strategy_of(banded)
result = [band_counts, join_name]
result

In [ ]:
check("5.9", result)

**5.10 Which orders have no customer?** Join *orders* to *customers* on *customer_id*.
* *inner_rows*: rows of the inner join
* *unmatched*: orders that match no customer (`left_anti`)

Put `[inner_rows, unmatched]` in *result*. Why are the guest orders missing from the inner join?

In [ ]:
inner_rows = orders.join(customers, "customer_id").count()
unmatched = orders.join(customers, "customer_id", "left_anti").count()
result = [inner_rows, unmatched]
result

In [ ]:
check("5.10", result)

---
Done? In the Theory part of Day 5, answer the red checklist topics aloud, with the strategies and shuffle numbers from this lab as evidence.